In [17]:
import pandas as pd
import numpy as np
import os
GTE_DIR=os.environ["GTE_DIR"]
from glaciation_time_estimator.auxiliary_func.config_reader import read_config
from glaciation_time_estimator.auxiliary_func.chunking_data import ChunkLoader
from glaciation_time_estimator.auxiliary_func.Helper_fun import float_range

ImportError: cannot import name 'float_range' from 'glaciation_time_estimator.auxiliary_func.Helper_fun' (/net/ch4/wolke_scratch/dnikolo/Glaciation_time_estimator/src/glaciation_time_estimator/auxiliary_func/Helper_fun.py)

In [2]:
config = read_config(
    os.path.join('/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/config_n2o.yaml'))
analyze_year=True
# years=[2007,2008,2009,2010,2011,2012,2013,2014,2015]
# years = [year for year in range(2007, 2016)]
years = [2009]
glac_threshold=config["glac_threshold"]

In [3]:
dataset = ChunkLoader(years,config=config)

In [4]:
print(dataset)

Dataset chunk loader
                 Analyzing files: {2009: '/wolke_scratch/dnikolo/Final_results/2009_all.parquet'}
                Currently loaded: 2009


In [5]:
def calculate_temp_occurance_rate(dfs):
    combined_cloud_df , _ , _ = dfs
    return combined_cloud_df.groupby("min_temp")["is_glaciating"].mean()*100

In [11]:
[i for i,b in enumerate(np.arange(0.2,0.51,0.1))]

[0, 1, 2, 3]

In [15]:
def float_range(start,stop,step):
    factor = 1/step
    return np.arange(start*factor,stop*factor,1)/factor

In [ ]:
glac_threshold_arr = [i for i in float_range(0.1,0.51,0.1)]
temp_occurance_dict = {}
for i,glac_threshold in enumerate(glac_threshold_arr):
    config = read_config(
    os.path.join(f'/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/glac_configs/n2o_2009_thresh_{int(glac_threshold*10):02}.yaml'))
    years = [2009]
    dataset = ChunkLoader(years,config=config)
    temp_occurance_dict[glac_threshold] = dataset.execute_analysis(calculate_temp_occurance_rate, cloud_columns=["min_temp"], glac_columns=[])
    

Loading years: 100%|██████████| 1/1 [00:00<00:00, 11.98year/s]


In [31]:
import pandas as pd

# Suppose temp_occurance_dict is defined like this:
# {
#   0.2: [<Series index=min_temp, values=is_glaciating>],
#   0.3: [<Series index=min_temp, values=is_glaciating>],
#   ...
# }

# Step 1: unwrap each one‐element list to get to the actual Series
series_dict = {
    glac_thresh: series_list[0]
    for glac_thresh, series_list in temp_occurance_dict.items()
}

# Step 2: build the DataFrame from the dict of Series
df = pd.DataFrame(series_dict)

# Step 3: label the axes
df.index.name = 'min_temp'
df.columns.name = 'glac_threshold'

# Step (2): stack it so that you get a Series whose index is (min_temp, glac_threshold):
ser = df.stack()
# Now ser is a pd.Series with a MultiIndex: (min_temp, glac_threshold), and values = is_glaciating.

# Step (3): turn that into a DataFrame and reset_index:
df_long = ser.reset_index()
df_long.columns = ['min_temp', 'glac_threshold', 'occurance_rate']

# Step (4): add a “year” column (here 2009):
df_long['year'] = 2009


df_long.to_csv("glac_thresh_occurance_rates.csv")
print(df_long)


    min_temp  glac_threshold  occurance_rate  year
0        -36             0.2       31.465838  2009
1        -36             0.3       17.785346  2009
2        -36             0.4       10.121308  2009
3        -36             0.5        5.863809  2009
4        -30             0.2       18.729856  2009
5        -30             0.3       10.318713  2009
6        -30             0.4        5.849524  2009
7        -30             0.5        3.426135  2009
8        -24             0.2       11.374272  2009
9        -24             0.3        6.146662  2009
10       -24             0.4        3.450529  2009
11       -24             0.5        2.034238  2009
12       -18             0.2        7.010120  2009
13       -18             0.3        3.745883  2009
14       -18             0.4        2.134780  2009
15       -18             0.5        1.289744  2009
16       -12             0.2        2.579294  2009
17       -12             0.3        1.261177  2009
18       -12             0.4   